In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql import types as T

from datetime import date

SEED = 42

START_DATE = "2024-01-01"
END_DATE = "2025-12-31"

BASE_PATH = "/Volumes/workspace/finance_analytics/finance_raw"

print("Finance source-data generator")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Output path: {BASE_PATH}")

In [0]:
%python
products_data = [
    (1, "Current Account", "Account", 0.00, 0.00),
    (2, "Premium Current Account", "Account", 0.50, 120.00),
    (3, "Savings Account", "Account", 2.50, 0.00),
    (4, "Personal Loan", "Loan", 7.50, 100.00),
    (5, "Auto Loan", "Loan", 6.25, 150.00),
    (6, "Home Mortgage", "Loan", 4.25, 250.00),
    (7, "Student Loan", "Loan", 5.00, 50.00),
    (8, "Business Loan", "Loan", 8.25, 200.00),
    (9, "Fixed Deposit", "Investment", 3.75, 0.00),
    (10, "Credit Builder", "Loan", 9.50, 75.00)
]

products_schema = T.StructType([
    T.StructField("product_id", T.IntegerType(), False),
    T.StructField("product_name", T.StringType(), False),
    T.StructField("product_type", T.StringType(), False),
    T.StructField("interest_rate", T.DoubleType(), False),
    T.StructField("annual_fee", T.DoubleType(), False)
])

products_df = spark.createDataFrame(products_data, products_schema)

display(products_df)

In [0]:
%python
branches_data = [
    (1, "Dublin Central", "Dublin", "Leinster", "Ireland"),
    (2, "Dublin North", "Dublin", "Leinster", "Ireland"),
    (3, "Dublin South", "Dublin", "Leinster", "Ireland"),
    (4, "Cork Central", "Cork", "Munster", "Ireland"),
    (5, "Cork North", "Cork", "Munster", "Ireland"),
    (6, "Limerick Central", "Limerick", "Munster", "Ireland"),
    (7, "Limerick East", "Limerick", "Munster", "Ireland"),
    (8, "Galway Central", "Galway", "Connacht", "Ireland"),
    (9, "Waterford Central", "Waterford", "Munster", "Ireland"),
    (10, "Kilkenny Central", "Kilkenny", "Leinster", "Ireland"),
    (11, "London Central", "London", "England", "UK"),
    (12, "Manchester Central", "Manchester", "England", "UK"),
    (13, "Birmingham Central", "Birmingham", "England", "UK"),
    (14, "Liverpool Central", "Liverpool", "England", "UK"),
    (15, "Leeds Central", "Leeds", "England", "UK"),
    (16, "Glasgow Central", "Glasgow", "Scotland", "UK"),
    (17, "Edinburgh Central", "Edinburgh", "Scotland", "UK"),
    (18, "Cardiff Central", "Cardiff", "Wales", "UK"),
    (19, "Belfast Central", "Belfast", "Northern Ireland", "UK"),
    (20, "Bristol Central", "Bristol", "England", "UK"),
    (21, "Milan Central", "Milan", "Lombardy", "Italy"),
    (22, "Rome Central", "Rome", "Lazio", "Italy"),
    (23, "Turin Central", "Turin", "Piedmont", "Italy"),
    (24, "Paris Central", "Paris", "Ile-de-France", "France"),
    (25, "Lyon Central", "Lyon", "Auvergne-Rhone-Alpes", "France"),
    (26, "Berlin Central", "Berlin", "Berlin", "Germany"),
    (27, "Munich Central", "Munich", "Bavaria", "Germany"),
    (28, "Madrid Central", "Madrid", "Madrid", "Spain"),
    (29, "Barcelona Central", "Barcelona", "Catalonia", "Spain"),
    (30, "Amsterdam Central", "Amsterdam", "North Holland", "Netherlands")
]

branches_schema = T.StructType([
    T.StructField("branch_id", T.IntegerType(), False),
    T.StructField("branch_name", T.StringType(), False),
    T.StructField("city", T.StringType(), False),
    T.StructField("region", T.StringType(), False),
    T.StructField("country", T.StringType(), False)
])

branches_df = spark.createDataFrame(branches_data, branches_schema)

display(branches_df)

In [0]:
%python
customers_df = (
    spark.range(1, 10001)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "first_name",
        F.element_at(
            F.array(
                F.lit("Aarav"),
                F.lit("Emma"),
                F.lit("Liam"),
                F.lit("Sofia"),
                F.lit("Noah"),
                F.lit("Olivia"),
                F.lit("Ethan"),
                F.lit("Mia"),
                F.lit("Arjun"),
                F.lit("Grace")
            ),
            (((F.col("customer_id") - 1) % 10) + 1).cast("int")
        )
    )
    .withColumn("last_name", F.concat(F.lit("Customer_"), F.col("customer_id")))
    .withColumn(
        "country",
        F.element_at(
            F.array(
                F.lit("Ireland"),
                F.lit("UK"),
                F.lit("Italy"),
                F.lit("France"),
                F.lit("Germany"),
                F.lit("Spain")
            ),
            ((F.col("customer_id") % 6) + 1).cast("int")
        )
    )
    .withColumn(
        "customer_segment",
        F.when(F.rand(SEED) < 0.15, "Premium")
         .when(F.rand(SEED + 1) < 0.50, "Standard")
         .otherwise("Basic")
    )
    .withColumn(
        "credit_score",
        F.floor(F.rand(SEED + 2) * 301 + 500).cast("int")
    )
    .withColumn(
        "customer_since",
        F.date_add(
            F.to_date(F.lit("2018-01-01")),
            F.floor(F.rand(SEED + 3) * 2922).cast("int")
        )
    )
)

display(customers_df.limit(20))

In [0]:
%python
print("Customer count:", customers_df.count())

customers_df.groupBy("customer_segment").count().show()

In [0]:
accounts_df = (
    spark.range(1, 15001)
    .withColumnRenamed("id", "account_id")
    .withColumn(
        "customer_id",
        (F.floor(F.rand(SEED + 10) * 10000) + 1).cast("int")
    )
    .withColumn(
        "product_id",
        F.expr("""
            CASE
                WHEN rand(20) < 0.50 THEN 1
                WHEN rand(21) < 0.70 THEN 2
                WHEN rand(22) < 0.90 THEN 3
                ELSE 9
            END
        """).cast("int")
    )
    .withColumn(
        "branch_id",
        (F.floor(F.rand(SEED + 11) * 30) + 1).cast("int")
    )
    .withColumn(
        "account_status",
        F.when(F.rand(SEED + 12) < 0.92, "Active")
         .when(F.rand(SEED + 13) < 0.50, "Closed")
         .otherwise("Dormant")
    )
    .withColumn(
        "open_date",
        F.date_add(
            F.to_date(F.lit("2020-01-01")),
            F.floor(F.rand(SEED + 14) * 2192).cast("int")
        )
    )
    .withColumn(
        "current_balance",
        F.round(
            F.rand(SEED + 15) * 50000,
            2
        )
    )
)

display(accounts_df.limit(20))

In [0]:
print("Account count:", accounts_df.count())

In [0]:
accounts_df.groupBy("customer_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(10)

In [0]:
invalid_accounts = (
    accounts_df
    .join(
        customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print("Accounts with invalid customer IDs:", invalid_accounts.count())

In [0]:
transactions_df = (
    spark.range(1, 500001)
    .withColumnRenamed("id", "transaction_id")
    .withColumn(
        "account_id",
        (F.floor(F.rand(SEED + 20) * 15000) + 1).cast("int")
    )
    .withColumn(
        "transaction_date",
        F.expr("""
            date_add(
                DATE '2024-01-01',
                CAST(floor(rand(23) * 731) AS INT)
            )
        """)
    )
    .withColumn(
        "transaction_type",
        F.when(F.rand(SEED + 24) < 0.55, "Debit")
         .otherwise("Credit")
    )
    .withColumn(
        "amount",
        F.round(
            F.rand(SEED + 25) * 2000 + 5,
            2
        )
    )
    .withColumn(
        "merchant_category",
        F.element_at(
            F.array(
                F.lit("Groceries"),
                F.lit("Transport"),
                F.lit("Utilities"),
                F.lit("Dining"),
                F.lit("Shopping"),
                F.lit("Healthcare"),
                F.lit("Entertainment"),
                F.lit("Travel")
            ),
            (F.floor(F.rand(SEED + 26) * 8) + 1).cast("int")
        )
    )
    .withColumn(
        "channel",
        F.element_at(
            F.array(
                F.lit("Online"),
                F.lit("Mobile"),
                F.lit("Branch"),
                F.lit("ATM")
            ),
            (F.floor(F.rand(SEED + 27) * 4) + 1).cast("int")
        )
    )
    .withColumn("currency", F.lit("EUR"))
)

In [0]:
print("Transaction count:", transactions_df.count())

In [0]:
display(
    transactions_df
    .orderBy("transaction_date")
    .limit(20)
)

In [0]:
transactions_df.select(
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date")
).show()

In [0]:
invalid_transactions = (
    transactions_df
    .join(
        accounts_df.select("account_id"),
        on="account_id",
        how="left_anti"
    )
)

print(
    "Transactions with invalid account IDs:",
    invalid_transactions.count()
)

In [0]:
transactions_df.groupBy("transaction_type") \
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("average_amount")
    ) \
    .show()